#The Transformer Block — Assembly Day

Today, we combine everything from the last four days into a single, reusable Transformer Block. This is the fundamental unit of models like GPT and BERT.

A standard Transformer block consists of two main parts:

1. **Multi-Head Attention:** Where the model looks at different parts of the sentence.

2. **Feed-Forward Network (FFN):** A small fully connected network that processes the information gained from attention.

**The Secret Sauce: LayerNorm and Residual Connections**
To prevent the model from forgetting the original input (and to help with training deep networks), we use Residual Connections (Add) and Layer Normalization (Norm).

1. **Implementing a Transformer Encoder Block**

In modern Keras, we can create this as a custom class for maximum flexibility.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        # 1. Multi-Head Attention
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        
        # 2. Feed-Forward Network
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"), 
            layers.Dense(embed_dim)
        ])
        
        # 3. Layer Normalization & Dropout
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=True):
        # Multi-Head Attention + Residual Connection
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        # Feed-Forward + Residual Connection
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Example Usage
sample_input = tf.random.uniform((32, 50, 128)) # (Batch, Seq, Dim)
transformer_block = TransformerBlock(embed_dim=128, num_heads=4, ff_dim=512)
output = transformer_block(sample_input)
print("Transformer Output Shape:", output.shape)

Transformer Output Shape: (32, 50, 128)
